# Practical 12 — Explainability and Interpretability

This notebook is the Colab-friendly counterpart to the agentic Claude Code / Cline
project. There, an agent runs `/explain <applicant>`; here we drive the same
deterministic reference tools directly. The pipeline is **attribute → check → draft**:
we attribute a logistic credit decision to its features using exact linear SHAP
(`phi_i = w_i * (x_i - baseline_i)`), verify the attribution is faithful via the
additivity gate (`sum_i phi_i + f(baseline) == f(x)`), and — for a denial — ground an
adverse-action notice in the top negative features. Everything is offline on bundled
fictional data; the tools do all the math, never the language model.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "12-xai-explainability"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## The bundled model and applicants

The model (`data/model.json`) is a fictional logistic credit scorecard: an intercept
plus a weight and a reference baseline for each feature. The applicants
(`data/applicants/*.json`) each carry a `features` block (the only thing the model
reads) and a `protected` block (age, sex, ...) that must never enter an explanation.
Let us load the model and inspect its feature table.

In [ ]:
import json
from tools._common import load_model, load_applicant, feature_arrays, applicant_vector

model = load_model()
keys, weights, baselines = feature_arrays(model)
print("Model:", model["name"])
print("Task :", model["task"])
print("Intercept:", model["intercept"], " threshold:", model["decision_threshold"])
print()
print(f"{'feature':<22}{'weight':>9}{'baseline':>10}")
for k, w, b in zip(keys, weights, baselines):
    print(f"{k:<22}{w:>9}{b:>10}")

## Step 1 — Attribute the decision (exact linear SHAP)

`attribute_applicant` loads the model + applicant, computes each feature's contribution
`phi_i = w_i * (x_i - baseline_i)` in log-odds space, and sorts the contributions most
creditworthiness-reducing first. We start with **alice**, who is denied.

In [ ]:
from tools.attribute import attribute_applicant, top_negative

alice = attribute_applicant("alice")
print("Applicant:", alice["applicant_id"], "|", alice["requested"])
print(f"decision = {alice['decision'].upper()}   prob = {alice['prob']:.4f}"
      f"   (baseline prob = {alice['baseline_prob']:.4f}, threshold = {alice['threshold']})")
print(f"\n{'feature':<22}{'value':>8}{'baseline':>10}{'phi':>10}")
for c in alice["contributions"]:
    print(f"{c['key']:<22}{c['value']:>8}{c['baseline']:>10}{c['phi']:>10.4f}")

## Step 2 — Check additivity (the faithfulness gate)

An attribution is only trustworthy if it reconstructs the model exactly:
`sum(phi) + f(baseline) == f(x)`. `check_additivity` reduces "is this explanation
faithful?" to a single residual gap, which for a linear model is ~0 (machine epsilon).
If the gap exceeded the tolerance, no notice would be written.

In [ ]:
from tools.check import check_additivity

chk = check_additivity(alice)
print(json.dumps(chk, indent=2))
assert chk["ok"], "additivity gate failed — explanation is not faithful"

## Step 3 — Draft a grounded adverse-action notice

Because alice is denied, we pull the top negative features — the factors that most
lowered her odds — and cite their pre-registered `adverse_action_reason` strings. This
is the part the LLM would narrate, but every reason is anchored to an exact `phi`, not
invented. Notice the **dominant** reason for alice is recent delinquencies.

In [ ]:
def draft_notice(result, k=4):
    if result["decision"] == "approve":
        return f"APPROVAL — {result['applicant_id']}: application approved (no adverse action)."
    lines = [f"ADVERSE ACTION NOTICE — applicant {result['applicant_id']}",
             "Your application was declined. Principal reasons, most significant first:"]
    for i, c in enumerate(top_negative(result, k=k), 1):
        lines.append(f"  {i}. {c['adverse_action_reason']}  (impact {c['phi']:+.3f})")
    return "\n".join(lines)

print(draft_notice(alice))

## Try it — carol: denied for a *different* dominant reason

The README suggests comparing alice and carol: both are denied, but carol's dominant
factor is high credit **utilization** (0.88 vs a 0.30 baseline), not delinquencies.
Same pipeline, different leading reason — the attribution, not the model, decides which.

In [ ]:
carol = attribute_applicant("carol")
print("carol decision:", carol["decision"], f"(prob {carol['prob']:.4f})")
print("carol top negative factor:", top_negative(carol, k=1)[0]["key"])
print("alice top negative factor:", top_negative(alice, k=1)[0]["key"])
print()
print(draft_notice(carol))

## Try it — bob: a clean applicant who is approved

The agent must not invent denial reasons when none exist. bob clears the threshold, so
the only honest output is an approval note.

In [ ]:
bob = attribute_applicant("bob")
print(check_additivity(bob)["ok"], "additivity ok")
print(f"bob: decision = {bob['decision'].upper()}  prob = {bob['prob']:.4f}"
      f"  (> baseline {bob['baseline_prob']:.4f})")
print(draft_notice(bob))

## Try it — protected attributes never enter the computation

alice's file carries a `protected` block (age, sex, marital status, national origin).
The model vector is built only from the model's own feature keys, so protected fields
are silently ignored — they appear in neither the attribution nor the notice.

In [ ]:
alice_raw = load_applicant("alice")
print("Protected fields present in the data file:", list(alice_raw["protected"].keys()))
x = applicant_vector(model, alice_raw)
print("Model vector length:", len(x), "== number of model features:", len(model["features"]))
attributed_keys = {c["key"] for c in alice["contributions"]}
leaked = attributed_keys & set(alice_raw["protected"].keys())
print("Protected keys that leaked into the attribution:", leaked or "none")
assert not leaked

## Recap / next steps

We ran the practical's reference tools end to end, fully offline:

1. **Attribute** (`tools/attribute.py`) — exact linear-SHAP contributions
   `phi_i = w_i * (x_i - baseline_i)` plus the approve/deny decision.
2. **Check** (`tools/check.py`) — the additivity gate: `sum(phi) + f(baseline) == f(x)`
   to ~machine epsilon, certifying the explanation is faithful.
3. **Draft** — a notice grounded in the top negative features (denials) or an approval.

Key takeaways: additivity holds for *any* linear model (try editing a weight in
`data/model.json` and re-running `check_additivity` — the gap stays ~0); the dominant
denial reason is a property of the attribution (delinquencies for alice, utilization for
carol); and protected attributes never reach the model. Next, add your own applicant
JSON to `data/applicants/` and re-run the three steps, or run `python -m pytest -q`.